In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.layers import Input, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D

dataframe = pd.read_csv("Electric_Production.csv")
dataframe.columns = dataframe.columns.str.strip()
dataframe["DATE"] = pd.to_datetime(dataframe["DATE"], format="%d-%m-%Y")
dataframe = dataframe.sort_values("DATE")

values = dataframe["Value"].values.reshape(-1, 1)

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(values)

window_size = 12

def build_sequences(series, window):
    features, targets = [], []
    for idx in range(len(series) - window):
        features.append(series[idx:idx + window])
        targets.append(series[idx + window])
    return np.array(features), np.array(targets)

X, y = build_sequences(scaled_values, window_size)

cutoff = int(0.8 * len(X))

X_train, X_test = X[:cutoff], X[cutoff:]
y_train, y_test = y[:cutoff], y[cutoff:]

def assess(model, label):
    predictions = model.predict(X_test)

    actual = scaler.inverse_transform(y_test)
    predicted = scaler.inverse_transform(predictions)

    error = np.sqrt(mean_squared_error(actual, predicted))
    print(f"{label} RMSE: {error}")

    plt.figure()
    plt.plot(actual, label="Actual")
    plt.plot(predicted, label="Predicted")
    plt.title(label)
    plt.legend()
    plt.show()

rnn_model = Sequential([
    SimpleRNN(50, activation="tanh", input_shape=(window_size, 1)),
    Dense(1)
])

rnn_model.compile(optimizer="adam", loss="mse")
rnn_model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0)

assess(rnn_model, "Simple RNN")

lstm_model = Sequential([
    LSTM(64, activation="tanh", input_shape=(window_size, 1)),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mse")
lstm_model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0)

assess(lstm_model, "LSTM")

gru_model = Sequential([
    GRU(64, activation="tanh", input_shape=(window_size, 1)),
    Dense(1)
])

gru_model.compile(optimizer="adam", loss="mse")
gru_model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0)

assess(gru_model, "GRU")

inp = Input(shape=(window_size, 1))

z = LayerNormalization()(inp)
z = MultiHeadAttention(num_heads=2, key_dim=16)(z, z)
z = Dropout(0.1)(z)

z = GlobalAveragePooling1D()(z)
out = Dense(1)(z)

transformer_model = tf.keras.Model(inp, out)

transformer_model.compile(optimizer="adam", loss="mse")
transformer_model.fit(X_train, y_train, epochs=10, batch_size=16, verbose=0)

assess(transformer_model, "Transformer")